# 6. The notebook configuration widget

Everything `configure()` does can also be done from a form inside the
notebook. The widget keeps named configurations, called **profiles**, between
sessions; it can test a connection and run a small real job before you commit
to a setting; and it handles the first-connection host key check with a
button.

A live widget cannot be embedded in static documentation, so this page shows
screenshots. They were taken from a real Jupyter session against the same
single-node test cluster as tutorials 3 to 5 (`clx@127.0.0.1:2222`), by
`scripts/render_widget_screenshots.py`, which presses the widget's own
buttons. The code cells below were executed.

## Opening it

Install the extra (`pip install "clustrix[widget]"`), then load the extension
once per notebook:

In [1]:
%load_ext clustrix

This registers the magics and puts `cluster`, `configure` and `get_config` in
the notebook's namespace. It does not display anything.

The widget appears when a cell starts with `%%remote`. Anything else in that
cell is ordinary code and runs after the widget is shown:

```python
%%remote
```

(From Python, `clustrix.display_modern_widget()` does the same.) On first use
it looks like this:

<img src="../_static/widget/first-run.png" alt="The widget on first run, with the Local profile selected" width="700">

From top to bottom:

- the **header** says in words what `@cluster` is doing right now ("currently
  runs on this machine") and carries the first of two indicators;
- **Profile** selects, renames, duplicates and deletes profiles;
- **Where to run** chooses the backend and the resources of one job;
- the **action row** has *Advanced settings*, *Test connection* (for backends
  that connect somewhere), *Run test job* and *Apply*;
- **Result** shows the outcome of the last action and carries the second
  indicator.

## Draft, Apply and Revert

The widget keeps three things apart:

1. what `@cluster` is using now (the live configuration);
2. what each profile holds on disk;
3. what is on the screen: a **draft** of the selected profile.

Editing a field changes only the draft. Nothing a job would use changes until
you press **Apply**, which saves the draft to its profile and makes it the
live configuration in the same step. **Revert** discards the draft and shows
the profile as saved. A draft survives switching to another profile and back.

As soon as a field differs from the saved profile, a line appears under the
profile row: "Edited. **Apply** saves these changes to the profile and uses
them; **Revert** discards them."

## The two indicators

- **Top right, in the header**: `in use` (green) when what is on screen is
  exactly what `@cluster` is using, otherwise `not applied` (grey). If you
  have edited anything, or selected another profile, it reads `not applied`
  until you press Apply.
- **Next to Result**: how the last button press ended, for example
  `connected`, `job completed`, `applied`, `invalid`, `unknown host`,
  `not connected`, `job failed`.

They answer different questions. A green `job completed` with a grey
`not applied` means: this configuration works, and you are not using it yet.

## SSH

Choosing **SSH** adds the connection section. The built-in profile
"SSH remote machine" is a template to fill in.

<img src="../_static/widget/ssh.png" alt="The SSH connection section filled in for the test cluster" width="700">

- **Host**, **Port**, **Username**: as for `ssh`.
- **Password**: used for this session only and never written to disk. When
  you leave the field, what you typed is moved out of the form into memory:
  the field empties and its placeholder changes to `entered; kept in memory
  only`. That is deliberate. Jupyter can save the content of every form field
  into the `.ipynb` file, and a password must not end up there. Typing again
  replaces what is held. The field beside it takes the *name* of an
  environment variable that holds the password instead.
- **SSH key file**: leave blank to use your SSH agent and default keys. A
  leading `~` stands for your home directory; the picture shows a full path,
  which works as well.
- **Set up SSH key**: generates a key pair and installs the public key on the
  host, using the password above once. Tick *Replace a key that already
  works* only to rotate a key.
- **Directory on the host for job files**: `remote_work_dir`.

### Validation

Apply, Test connection and Run test job first check the form. Fields that
need fixing are outlined and listed, and nothing is saved or sent.

<img src="../_static/widget/validation.png" alt="Validation: host and username are missing, outlined in red" width="700">

### Test connection, and trusting a host

**Test connection** connects and authenticates without running anything. On a
computer that has never connected to the host, the result is the host key
check from tutorial 3, with the fingerprint and a button:

<img src="../_static/widget/unknown-host.png" alt="Unknown host: the fingerprint is shown with a Trust this host button" width="700">

Compare the fingerprint with the one your cluster's administrators publish.
If it matches, **Trust this host** records exactly the key that was shown in
`~/.ssh/known_hosts` and tries the connection again:

<img src="../_static/widget/trusted.png" alt="After Trust this host: connected" width="700">
 If it does not match,
stop and ask your administrators; do not press the button.

## SLURM

**SLURM** uses the same connection section, pointed at the login node, and
adds a **Partition** field. Leave it blank for the scheduler's default.

<img src="../_static/widget/slurm.png" alt="SLURM selected, with the partition field" width="700">

### Run test job

**Run test job** submits one small real job through exactly the path a
`@cluster` call takes: upload, environment, `sbatch`, wait, verified result.
It reports where the job ran, the Python it ran under, and the value it
returned.

<img src="../_static/widget/test-job.png" alt="A test job completed on SLURM" width="700">

The first test job on a new host builds the remote environment and takes
correspondingly longer. If this works, your functions will be submitted the
same way.

### Apply

<img src="../_static/widget/applied.png" alt="After Apply: the header reads in use and names the backend" width="700">

After **Apply** the header names the backend in use and its indicator turns
to `in use`. From this moment, every `@cluster` call in the notebook goes
there. The profile on disk now holds these settings, without the password.

## Hugging Face

<img src="../_static/widget/huggingface.png" alt="The Hugging Face Jobs section" width="700">

**Namespace** is your user name or an organisation. **Hardware** is the
flavor. The **token** is used for this session and never saved, and like
the password it is moved out of the field once entered; leave it
blank to use `HF_TOKEN` or the login stored by `hf auth login`. GPU hardware
is billed per second and is refused unless the checkbox is ticked. See
[tutorial 7](07_huggingface_jobs.ipynb).

## Advanced settings

<img src="../_static/widget/advanced.png" alt="The Advanced section expanded" width="700">

| Field | Setting | Tutorial |
|---|---|---|
| Package manager | `package_manager` (`pip`, `uv`, ...) | |
| Python executable | `python_executable` | 3 |
| Existing conda environment | `conda_env_name`; blank builds an environment | 3 |
| Install my local packages on the worker | `replicate_local_environment` | 3 |
| Delete job files on the host after success | `cleanup_on_success` | 3 |
| Modules to load | `module_loads`, space-separated | 3, 4 |
| Environment variables | `plain_environment_variables`, `NAME=value` per line; shown as typed and saved in the profile, so nothing secret goes here | 3 |
| Secret environment variables | `environment_variables`, `NAME=value` per line; masked once you leave the field, this session only, never saved | 3 |
| Shell commands to run before the job | `pre_execution_commands`, one per line | 3 |
| Profiles file, Export, Import | see below | |

## Profiles

A profile is a named configuration. The built-in ones are templates.

- **Duplicate** copies the selected profile under a new name and selects the
  copy. Start here for each cluster you use.
- **Rename to**: type a new name in the field.
- **Delete** asks for a second click: the first only arms the button, and
  anything else you do disarms it. There is no undo.
- **Export** writes every profile to the file named under *Profiles file* in
  Advanced settings; **Import** replaces every profile with the ones in that
  file. That is how to move profiles to another computer. Passwords, tokens
  and secret environment variables are not in the file; the plain
  environment variables are.

Profiles live in `profiles.yml` in the clustrix configuration directory
(`~/.clustrix`, or `CLUSTRIX_CONFIG_DIR`). They can be read from Python too:

In [2]:
from clustrix import ProfileManager

manager = ProfileManager()
for name in manager.get_profile_names():
    print(name)

Local single-core
Local all cores
SLURM cluster
SSH remote machine
HuggingFace Jobs (CPU)
HuggingFace Jobs (GPU)


## The dark theme

The widget takes its colours from the notebook's theme.

<img src="../_static/widget/dark.png" alt="The widget in JupyterLab's dark theme" width="700">

## Without the widget: `%clustrix`

The line magic covers what needs no form.

In [3]:
%clustrix status
print("a host is configured:", get_config().cluster_host is not None)

backend      : local (no cluster host set; cluster_type is 'slurm')
resources    : 4 cores, 8GB, 01:00:00
a host is configured: False


A fresh installation says `local` even though the `cluster_type` field's
default is `slurm`: with no host set, calls run on this machine, as tutorial 1
explains, and `status` reports the backend in effect.

`%clustrix status` prints and returns nothing. The other forms also *return*
the configuration object, and when one of them is the last line of a cell the
notebook displays that object in full, which is long; end the line with `;`
or follow it with another to avoid that.

`%clustrix config <profile>` makes a saved profile the live configuration, as
Apply does:

In [4]:
%clustrix config Local all cores
print(get_config().cluster_type, get_config().default_cores)

✅ Applied profile 'Local all cores'
local -1


`%clustrix load <file>` loads a YAML or JSON configuration file, like
`clustrix.config.load_config`. A profile name that does not exist is refused
with the list of those that do:

In [5]:
%clustrix config Big cluster
print("still:", get_config().cluster_type)

❌ No profile named 'Big cluster'. Available: HuggingFace Jobs (CPU), HuggingFace Jobs (GPU), Local all cores, Local single-core, SLURM cluster, SSH remote machine
still: local


## If something looks wrong

| You see | It means |
|---|---|
| `%%remote` prints how to install the widget | `ipywidgets` is missing. `pip install "clustrix[widget]"` and restart the kernel. |
| The header says `not applied` although the test job passed | Testing does not apply. Press **Apply**. |
| `invalid`, with outlined fields | The form is incomplete or a value is malformed; the Result box lists each problem. |
| `unknown host` | First connection from this computer. Verify the fingerprint, then **Trust this host**. |
| `not connected`, `No such file or directory` naming a key | The file in **SSH key file** does not exist. Check the path. |
| The password field is empty right after I typed in it | It was taken: the placeholder reads `entered; kept in memory only`. |
| `not connected`, authentication failed | Wrong user name or key, or the public key is not on the host. **Set up SSH key** installs one. |
| My jobs still go to the old place | Another profile is live. The header always names what `@cluster` is using. |
| The password is gone after a restart | By design. Passwords and tokens are kept for the session only. |

## Next

[Tutorial 7](07_huggingface_jobs.ipynb): Hugging Face Jobs.